# Line Following Autonomous Driving with OpenCV

In this tutorial, we'll use basic functionalities of OpenCV to detect yellow lines (default color) in the image and control the direction of the chassis based on the position of these lines. Please note that in this example, the chassis won't move. Instead, we'll only showcase the algorithms using OpenCV on the image. For safety reasons, we won't integrate motion control in this tutorial, as it's heavily influenced by external factors. Users should fully understand the code's functionality before adding corresponding motion control features.

If you want to control the robot's movement through this example, please refer to the "Python Chassis Motion Control" section to add the relevant motion control functions (our open-source example is located in robot_ctrl.py).

## Preparation
Since the product automatically runs the main program on startup, and the main program occupies the camera resources, this tutorial cannot be used in that state. You need to stop the main program or disable its automatic startup before restarting the robot.

It should be noted that because the robot's main program uses multithreading and is configured to start automatically via a service, the usual method of `sudo killall python` often does not work. Therefore, we will introduce a method to disable the main program's automatic startup.

If you have already disabled the automatic startup of the robot's main program, you do not need to perform the following "Stopping the Main Program" section.

### Stopping the Main Program
1. Click the "" icon next to the tab at the top of this page to open a new tab named Launcher.
2. Click Terminal under Other to open the terminal window.
3. In the terminal window, type `bash` and press Enter.
4. You can now use the Bash Shell to control the robot.
5. Enter the command: `systemctl --user stop ugv-app.service`.
6. On the terminal page, after the command has been executed, continue with the remaining part of this tutorial.

## Example

The following code block can be run directly:

1. Select the code block below.
2. Press Shift + Enter to run the code block.
3. Watch the real-time video window.
4. Press STOP to close the real-time video and release the camera resources.

### If you cannot see the real-time camera feed when running:

- Click on Kernel -> Shut down all kernels above.
- Close the current section tab and open it again.
- Click `STOP` to release the camera resources, then run the code block again.
- Reboot the device.

### Features of this Section

After running the following code block, you can place the yellow tape in front of the camera and observe whether there are any dots drawn on the outline of the yellow tape in the ROI area of ​​the image.

In [ ]:
import cv2  # Import the OpenCV library for image processing
import imutils, math  # Auxiliary libraries for image processing and mathematical operations
from picamera2 import Picamera2  # Library for accessing the Raspberry Pi Camera
import numpy as np
from IPython.display import display, Image  # Library for displaying images in Jupyter Notebook
import ipywidgets as widgets  # Library for creating interactive widgets such as buttons
import threading  # Library for creating new threads to execute tasks asynchronously

# Stop button
# ================
stopButton = widgets.ToggleButton(
    value=False,
    description='STOP',
    disabled=False,
    button_style='danger',  # Button style: 'success', 'info', 'warning', 'danger', or ''
    tooltip='Description',
    icon='square'  # FontAwesome icon name (without the `fa-` prefix)
)

# findline autodrive
roi = [
    (250, 300, 40, 600, 0.1),
    (300, 400, 40, 600, 0.3),
    (400, 480, 40, 600, 0.6),
]

# Target line color, in HSV color space
line_lower = np.array([0, 100, 187])
line_upper = np.array([255, 255, 255])

def view(button):
    # If you are using a CSI camera, uncomment the picam2 related code below, 
    # and comment out the camera related code.
    # This is because the latest version of OpenCV (4.9.0.80) no longer supports CSI cameras, 
    # and you need to use picamera2 to capture camera images.
    
    # picam2 = Picamera2()
    # picam2.configure(picam2.create_video_configuration(main={"format": 'XRGB8888', "size": (640, 480)}))
    # picam2.start()

    camera = cv2.VideoCapture(0) 
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    
    display_handle=display(None, display_id=True)
    
    while True:
        # img = picam2.capture_array()
        _, img = camera.read()
        # frame = cv2.flip(frame, 1) # if your camera reverses your image

        # frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        cx = w // 2

        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

        weight_sum = 0
        cx_sum = 0
        has_line = False

        input_speed = 0.0
        input_turning = 0.0

        for (y1, y2, x1, x2, wt) in roi:
            crop = lab[y1:y2, x1:x2]
            mask = cv2.inRange(crop, line_lower, line_upper)
            cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not cnts:
                continue

            c = max(cnts, key=cv2.contourArea)
            if cv2.contourArea(c) < 50:
                continue

            (x, y), _, _ = cv2.minAreaRect(c)
            x += x1
            cx_sum += x * wt
            weight_sum += wt
            has_line = True

            cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 1)
            cv2.circle(img, (int(x), int((y1+y2)/2)), 5, (0,0,255), -1)

        if has_line:
            x_avg = int(cx_sum / weight_sum)
            cv2.circle(img, (x_avg, h-50), 8, (0,255,255), -1)

        _, frame = cv2.imencode('.jpeg', line_mask)
        display_handle.update(Image(data=frame.tobytes()))
        if stopButton.value==True:
            # picam2.close() # If yes, close the camera
            camera.release() # If yes, close the camera
            display_handle.update(None)


# Display the "STOP" button and start a thread to execute the display function
# ================
display(stopButton)
thread = threading.Thread(target=view, args=(stopButton,))
thread.start()